# Mini-Projeto 01 · Conselheiro Shinobi
## Sistema baseado em regras para situações de combate em Sekiro

**Autor:** Gabriel Rafá Martins Freire  
**Matrícula:** 20230145310  


## 1. Domínio e escopo

O **Conselheiro Shinobi** é um Sistema Baseado em Conhecimento (SBC) que
recomenda **uma ação imediata de combate** em *Sekiro: Shadows Die Twice*.
O usuário informa uma situação; o motor Experta aplica regras IF–THEN,
produz fatos intermediários e explica a recomendação.

O recorte inclui estocadas, varreduras, agarres, ataques comuns e uso da
Cabaça Curativa. Mikiri é uma resposta a estocadas, saltar é uma resposta
a varreduras e defletir é uma resposta a ataques comuns. O modelo usa uma
estratégia conservadora de afastamento para agarres e estocadas sem Mikiri.
Essa última escolha é uma simplificação: estocadas também podem ser
defletidas com o tempo correto, e movimentos específicos têm exceções.

**Premissas do modelo:** um jogador vivo, um adversário e uma situação por
execução; o tipo do ataque é informado manualmente. `nenhum` significa que
não há ataque em curso, mas só `abertura_segura=True` informa tempo e espaço
para usar a cura. O limiar de 30% e as prioridades são escolhas deste
projeto, não valores oficiais do jogo. Há apenas uma ação imediata por
consulta; uma nova situação exige nova execução.

| Campo de `Combate` | Tipo / valores | Significado |
|---|---|---|
| `cenario` | texto não vazio | Identificador da consulta |
| `vida` | número finito: `0 < vida <= 100` | Porcentagem de vida do jogador |
| `ataque` | `estocada`, `varredura`, `agarre`, `normal`, `nenhum` | Movimento observado |
| `mikiri` | `True` ou `False` | Habilidade Mikiri desbloqueada |
| `curas` | inteiro maior ou igual a zero | Cargas restantes da Cabaça Curativa |
| `abertura_segura` | `True` ou `False` | Janela segura para se curar; só pode ser `True` com `ataque="nenhum"` |

## 2. Encadeamento e modelagem dos fatos

| Etapa | Fato | Papel | Exemplo |
|---|---|---|---|
| Entrada | `Combate` | Dados informados | Estocada e Mikiri desbloqueado |
| Nível 1 | `Analise` | Interpretação do estado | R2 identifica ataque perigoso |
| Nível 2 | `Estrategia` | Ação candidata | R5 propõe Mikiri usando o fato de R2 |
| Nível 3 | `Recomendacao` | Decisão única | R12 seleciona a estratégia de R5 |

O caminho **R2 → R5 → R12** contém três disparos dependentes: R5 exige a
conclusão de R2, e R12 exige a conclusão de R5. Não são apenas três regras
independentes que consultam a entrada. A cura também encadeia os três níveis:
R1 e R4 produzem análises, R9 combina essas análises e R13 decide.

O campo `cenario` relaciona os fatos de uma consulta. O campo `cadeia` dos
fatos derivados armazena sua proveniência. O fato original não é alterado;
cada nível acrescenta uma conclusão com uma função distinta.

In [ ]:
%pip install --quiet experta==1.9.4

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yfinance 0.2.66 requires frozendict>=2.3.4, but you have frozendict 1.2 which is incompatible.


In [ ]:
import collections
import collections.abc
import math
import platform
from importlib.metadata import version

# Compatibilidade da dependência antiga frozendict com Python atual.
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping

from experta import KnowledgeEngine, Rule, Fact, MATCH, AS, P, OR, NOT

print("Python:", platform.python_version())
print("Experta:", version("experta"))
print("frozendict:", version("frozendict"))

Python: 3.13.15
Experta: 1.9.4
frozendict: 1.2


### 3. Classes de fatos e textos das ações

As quatro classes seguem a estrutura simples dos notebooks de exemplo.
Os dados entram em `Combate`; os demais fatos são criados apenas pelas
regras. `ACOES` contém textos para apresentação, sem decidir qual ação usar.

In [ ]:
class Combate(Fact):
    """Entrada: cenario, vida, ataque, mikiri, curas, abertura_segura."""
    pass

class Analise(Fact):
    """Nível 1: cenario, tipo e cadeia de regras que sustentam o fato."""
    pass

class Estrategia(Fact):
    """Nível 2: cenario, categoria, acao e cadeia."""
    pass

class Recomendacao(Fact):
    """Nível 3: cenario, acao, explicacao e cadeia."""
    pass

ACOES = {
    "MIKIRI": "Executar o Contra-ataque Mikiri no momento da estocada.",
    "SALTAR": "Saltar sobre a varredura e, se possível, saltar sobre o inimigo.",
    "AFASTAR": "Sair do alcance do ataque com movimentação adequada ao golpe.",
    "DEFLETIR": "Defletir o ataque comum no instante do impacto.",
    "CURAR": "Usar uma carga da Cabaça Curativa na abertura segura informada.",
    "BUSCAR_ABERTURA": "Reposicionar-se para buscar segurança; reavaliar a cura depois.",
    "OBSERVAR": "Observar o próximo movimento e preparar uma resposta.",
}

# Esta tabela identifica as regras finais para o relatório de conflito.
# A prioridade efetiva é lida dos decoradores @Rule, não desta tabela.
REGRA_POR_CATEGORIA = {"reacao": "R12", "cura": "R13", "cautela": "R14"}

## 4. Base de conhecimento: 14 regras

| Regra | Nível | `salience` | Regra em linguagem natural |
|---|---:|---:|---|
| R1 | 1 | 100 | SE a vida for menor ou igual a 30%, ENTÃO identificar vida baixa. |
| R2 | 1 | 99 | SE o ataque for estocada, varredura ou agarre, ENTÃO identificar ataque perigoso. |
| R3 | 1 | 98 | SE o ataque for normal, ENTÃO identificar ataque comum. |
| R4 | 1 | 97 | SE não houver ataque em curso, ENTÃO identificar ausência de ataque. |
| R5 | 2 | 90 | SE houver ataque perigoso do tipo estocada e Mikiri desbloqueado, ENTÃO propor MIKIRI como reação. |
| R6 | 2 | 89 | SE houver ataque perigoso do tipo varredura, ENTÃO propor SALTAR como reação. |
| R7 | 2 | 88 | SE houver ataque perigoso e ele for agarre, OU for estocada sem Mikiri, ENTÃO propor AFASTAR como reação. |
| R8 | 2 | 87 | SE houver ataque comum, ENTÃO propor DEFLETIR como reação. |
| R9 | 2 | 86 | SE houver vida baixa, ausência de ataque, abertura segura e ao menos uma carga de cura, ENTÃO propor CURAR. |
| R10 | 2 | 85 | SE houver vida baixa, ENTÃO propor BUSCAR_ABERTURA como alternativa de cautela. |
| R11 | 2 | 84 | SE houver ausência de ataque e NÃO houver vida baixa, ENTÃO propor OBSERVAR como cautela. |
| R12 | 3 | 30 | SE existir estratégia de reação e NÃO existir recomendação para o cenário, ENTÃO recomendar essa reação. |
| R13 | 3 | 20 | SE existir estratégia de cura e NÃO existir recomendação para o cenário, ENTÃO recomendar CURAR. |
| R14 | 3 | 10 | SE existir estratégia de cautela e NÃO existir recomendação para o cenário, ENTÃO recomendar essa cautela. |

As regras de análise e estratégia têm guardas `NOT` para evitar fatos
repetidos. Todas as decisões finais exigem ausência de recomendação.
Em R7, o agrupamento lógico é: ataque perigoso E (agarre OU estocada sem Mikiri).

## 5. Estratégia de resolução de conflitos

As regras de análise têm `salience` de 97 a 100; as de estratégia, de 84
a 90; as de decisão, 30, 20 e 10. Assim, a situação é analisada e todas as
estratégias aplicáveis são geradas antes da escolha final.

| Regra de decisão | Categoria | Prioridade |
|---|---|---:|
| R12 | Reagir a um ataque em curso | 30 |
| R13 | Curar em abertura segura | 20 |
| R14 | Cautela: buscar abertura ou observar | 10 |

Com vida baixa e uma estocada, R5 propõe Mikiri e R10 propõe buscar abertura.
R12 vence R14 por prioridade. Após a primeira recomendação ser declarada,
`NOT(Recomendacao(cenario=MATCH.c))` deixa de ser verdadeiro e bloqueia as
outras decisões. Com vida baixa e cura segura, R13 vence R14.

R10 gera uma alternativa mesmo quando há uma estratégia melhor. Isso é
intencional: torna a competição entre regras observável. R10 e R11 não
concorrem entre si, pois exigem condições opostas de vida baixa.

O sistema imprime as candidatas e suas prioridades, o trace completo e a
cadeia causal da vencedora. O trace inclui tudo que disparou; a explicação
causal inclui somente os fatos que sustentam a decisão, além do motivo da
prioridade. Não há ordenação de ações em Python para escolher a vencedora:
a escolha é feita pelos decoradores `@Rule` do Experta.

## 6. Motor de inferência e explicações

`registrar()` declara um fato e guarda o motivo, o nível, a prioridade e a
proveniência de cada disparo. `concluir()` é chamado pela regra final que
venceu; ele monta o texto e relata as estratégias candidatas. Esses métodos
apenas registram conclusões: as condições de decisão estão nas 14 regras.

`AS` vincula os fatos usados como antecedentes. A cadeia de cada conclusão
é construída a partir desses fatos, evitando uma explicação fixa que não
corresponda ao que o motor realmente executou.

In [ ]:
class ConselheiroSekiro(KnowledgeEngine):
    def __init__(self, exibir_trace=True):
        super().__init__()
        self.exibir_trace = exibir_trace
        self.trace = []
        self.conflitos = []

    def reset(self, **kwargs):
        self.trace = []
        self.conflitos = []
        super().reset(**kwargs)

    def prioridade(self, regra):
        return getattr(type(self), regra.lower()).salience

    def registrar(self, regra, fato, motivo, antecedentes=()):
        # Unir a proveniência dos antecedentes, sem repetir regras.
        origens = [r for a in antecedentes for r in a["cadeia"]]
        fato["cadeia"] = tuple(dict.fromkeys(origens + [regra]))
        self.declare(fato)
        nivel = {Analise: 1, Estrategia: 2, Recomendacao: 3}[type(fato)]
        evento = {
            "cenario": fato["cenario"], "regra": regra, "nivel": nivel,
            "salience": self.prioridade(regra), "motivo": motivo,
            "conclusao": fato.get("tipo", fato.get("acao")),
            "cadeia": fato["cadeia"],
        }
        self.trace.append(evento)
        if self.exibir_trace:
            print(f'[{regra} | nível {nivel} | salience {evento["salience"]}] '
                  f'{motivo} Conclusão: {evento["conclusao"]}.')

    def concluir(self, regra, estrategia):
        cenario = estrategia["cenario"]
        candidatas = []
        for fato in self.facts.values():
            if isinstance(fato, Estrategia) and fato["cenario"] == cenario:
                regra_final = REGRA_POR_CATEGORIA[fato["categoria"]]
                candidatas.append({
                    "acao": fato["acao"], "regra": regra_final,
                    "salience": self.prioridade(regra_final),
                })

        # O motor já escolheu a regra atual pela salience.
        # Esta lista apenas torna as concorrentes visíveis.
        evento = {"cenario": cenario, "candidatas": candidatas,
                  "vencedora": regra, "acao": estrategia["acao"]}
        self.conflitos.append(evento)
        if self.exibir_trace:
            print("Candidatas à decisão:")
            for item in candidatas:
                print(f'  {item["regra"]}: {item["acao"]} '
                      f'(salience={item["salience"]})')
            print(f"Selecionada pelo Experta: {regra}.")

        cadeia = estrategia["cadeia"] + (regra,)
        motivos = [e["motivo"] for e in self.trace
                   if e["cenario"] == cenario
                   and e["regra"] in estrategia["cadeia"]]
        criterio = (
            f'{regra} seleciona a categoria {estrategia["categoria"]} '
            f'com salience {self.prioridade(regra)}. '
            'A presença da recomendação bloqueia outras decisões pelo NOT.'
        )
        if len(candidatas) > 1:
            alternativas = [f'{x["regra"]} (salience {x["salience"]})'
                            for x in candidatas if x["regra"] != regra]
            criterio += ' Essa prioridade supera ' + ', '.join(alternativas) + '.'
        explicacao = (
            f'A ação {estrategia["acao"]} foi escolhida porque '
            + ', '.join(cadeia) + ' dispararam. '
            + ' '.join(motivos) + ' ' + criterio
        )
        self.registrar(
            regra,
            Recomendacao(cenario=cenario, acao=estrategia["acao"],
                         explicacao=explicacao),
            criterio, antecedentes=(estrategia,),
        )

    # NÍVEL 1: interpretar os dados de entrada.
    @Rule(Combate(cenario=MATCH.c, vida=P(lambda v: v <= 30)),
          NOT(Analise(cenario=MATCH.c, tipo="vida_baixa")), salience=100)
    def r1(self, c):
        self.registrar("R1", Analise(cenario=c, tipo="vida_baixa"),
                       "A vida está no limiar de 30% ou abaixo dele.")

    @Rule(Combate(cenario=MATCH.c,
                  ataque=P(lambda a: a in ("estocada", "varredura", "agarre"))),
          NOT(Analise(cenario=MATCH.c, tipo="ataque_perigoso")), salience=99)
    def r2(self, c):
        self.registrar("R2", Analise(cenario=c, tipo="ataque_perigoso"),
                       "O movimento pertence ao grupo de ataques perigosos.")

    @Rule(Combate(cenario=MATCH.c, ataque="normal"),
          NOT(Analise(cenario=MATCH.c, tipo="ataque_comum")), salience=98)
    def r3(self, c):
        self.registrar("R3", Analise(cenario=c, tipo="ataque_comum"),
                       "O movimento informado é um ataque comum.")

    @Rule(Combate(cenario=MATCH.c, ataque="nenhum"),
          NOT(Analise(cenario=MATCH.c, tipo="sem_ataque")), salience=97)
    def r4(self, c):
        self.registrar("R4", Analise(cenario=c, tipo="sem_ataque"),
                       "Não há ataque em curso no cenário informado.")

    # NÍVEL 2: propor estratégias usando as análises do nível anterior.
    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="ataque_perigoso"),
          Combate(cenario=MATCH.c, ataque="estocada", mikiri=True),
          NOT(Estrategia(cenario=MATCH.c, acao="MIKIRI")), salience=90)
    def r5(self, c, a):
        self.registrar("R5", Estrategia(cenario=c, categoria="reacao", acao="MIKIRI"),
                       "É uma estocada e Mikiri está desbloqueado.", (a,))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="ataque_perigoso"),
          Combate(cenario=MATCH.c, ataque="varredura"),
          NOT(Estrategia(cenario=MATCH.c, acao="SALTAR")), salience=89)
    def r6(self, c, a):
        self.registrar("R6", Estrategia(cenario=c, categoria="reacao", acao="SALTAR"),
                       "A varredura requer uma resposta por salto neste modelo.", (a,))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="ataque_perigoso"),
          OR(Combate(cenario=MATCH.c, ataque="agarre"),
             Combate(cenario=MATCH.c, ataque="estocada", mikiri=False)),
          NOT(Estrategia(cenario=MATCH.c, acao="AFASTAR")), salience=88)
    def r7(self, c, a):
        self.registrar("R7", Estrategia(cenario=c, categoria="reacao", acao="AFASTAR"),
                       "Há agarre ou estocada sem Mikiri; o modelo propõe afastamento.", (a,))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="ataque_comum"),
          NOT(Estrategia(cenario=MATCH.c, acao="DEFLETIR")), salience=87)
    def r8(self, c, a):
        self.registrar("R8", Estrategia(cenario=c, categoria="reacao", acao="DEFLETIR"),
                       "O ataque comum admite a estratégia de deflexão.", (a,))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="vida_baixa"),
          AS.b << Analise(cenario=MATCH.c, tipo="sem_ataque"),
          Combate(cenario=MATCH.c, curas=P(lambda x: x > 0), abertura_segura=True),
          NOT(Estrategia(cenario=MATCH.c, acao="CURAR")), salience=86)
    def r9(self, c, a, b):
        self.registrar("R9", Estrategia(cenario=c, categoria="cura", acao="CURAR"),
                       "Há vida baixa, carga de cura e abertura segura sem ataque.", (a, b))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="vida_baixa"),
          NOT(Estrategia(cenario=MATCH.c, acao="BUSCAR_ABERTURA")), salience=85)
    def r10(self, c, a):
        self.registrar("R10", Estrategia(cenario=c, categoria="cautela", acao="BUSCAR_ABERTURA"),
                       "A vida baixa também justifica uma alternativa de reposicionamento.", (a,))

    @Rule(AS.a << Analise(cenario=MATCH.c, tipo="sem_ataque"),
          NOT(Analise(cenario=MATCH.c, tipo="vida_baixa")),
          NOT(Estrategia(cenario=MATCH.c, acao="OBSERVAR")), salience=84)
    def r11(self, c, a):
        self.registrar("R11", Estrategia(cenario=c, categoria="cautela", acao="OBSERVAR"),
                       "Não há ataque nem classificação de vida baixa; cabe observar.", (a,))

    # NÍVEL 3: resolver o conflito e declarar UMA recomendação.
    @Rule(AS.e << Estrategia(cenario=MATCH.c, categoria="reacao"),
          NOT(Recomendacao(cenario=MATCH.c)), salience=30)
    def r12(self, c, e):
        self.concluir("R12", e)

    @Rule(AS.e << Estrategia(cenario=MATCH.c, categoria="cura"),
          NOT(Recomendacao(cenario=MATCH.c)), salience=20)
    def r13(self, c, e):
        self.concluir("R13", e)

    @Rule(AS.e << Estrategia(cenario=MATCH.c, categoria="cautela"),
          NOT(Recomendacao(cenario=MATCH.c)), salience=10)
    def r14(self, c, e):
        self.concluir("R14", e)

## 7. Validação da entrada e execução de uma consulta

A função abaixo segue `reset()`, `declare()` e `run()` dos exemplos.
As validações conferem formato e coerência da entrada; não escolhem ações.
Uma nova instância evita que fatos de um teste contaminem o seguinte.

O limite de 50 disparos protege a demonstração contra uma alteração que
introduza ciclos. Na base atual, os níveis só acrescentam fatos e as
guardas `NOT` impedem repetições; a execução termina naturalmente.

In [ ]:
def validar_cenario(dados):
    campos = {"cenario", "vida", "ataque", "mikiri", "curas", "abertura_segura"}
    if set(dados) != campos:
        raise ValueError(f"Informe exatamente estes campos: {sorted(campos)}")
    if not isinstance(dados["cenario"], str) or not dados["cenario"].strip():
        raise ValueError("cenario deve ser um texto não vazio.")
    vida = dados["vida"]
    if type(vida) not in (int, float) or not math.isfinite(vida) or not 0 < vida <= 100:
        raise ValueError("vida deve ser um número finito maior que 0 e até 100.")
    ataques = ("estocada", "varredura", "agarre", "normal", "nenhum")
    if not isinstance(dados["ataque"], str) or dados["ataque"] not in ataques:
        raise ValueError(f"ataque deve ser um destes valores: {ataques}")
    if type(dados["curas"]) is not int or dados["curas"] < 0:
        raise ValueError("curas deve ser um inteiro maior ou igual a zero.")
    for campo in ("mikiri", "abertura_segura"):
        if type(dados[campo]) is not bool:
            raise ValueError(f"{campo} deve ser True ou False.")
    if dados["abertura_segura"] and dados["ataque"] != "nenhum":
        raise ValueError("Abertura segura exige ausência de ataque em curso.")

def executar_cenario(dados, exibir=True):
    validar_cenario(dados)
    motor = ConselheiroSekiro(exibir_trace=exibir)
    motor.reset()
    if exibir:
        print(f'\nCENÁRIO: {dados["cenario"]}')
        print("Entrada:", dados)
    motor.declare(Combate(**dados))
    motor.run(steps=50)
    if motor.agenda.activations:
        raise RuntimeError("Limite de disparos atingido; revise as regras.")
    decisoes = [f for f in motor.facts.values() if isinstance(f, Recomendacao)]
    if len(decisoes) != 1:
        raise RuntimeError(f"Esperada uma recomendação; foram geradas {len(decisoes)}.")
    decisao = decisoes[0]
    if exibir:
        print("\nAÇÃO:", decisao["acao"])
        print(ACOES[decisao["acao"]])
        # Regras do mesmo nível são antecedentes paralelos, não uma cadeia entre si.
        etapas = []
        for nivel in (1, 2, 3):
            regras = [e["regra"] for e in motor.trace
                      if e["nivel"] == nivel and e["regra"] in decisao["cadeia"]]
            etapas.append(" + ".join(regras))
        print("CADEIA CAUSAL:", " → ".join(etapas))
        print("EXPLICAÇÃO:", decisao["explicacao"])
    return {"motor": motor, "decisao": decisao,
            "trace": motor.trace, "conflitos": motor.conflitos}

## 8. Três casos principais, com saídas esperadas

Os cenários são sintéticos e verificam a base de regras; não são dados
coletados de partidas. Cada teste compara a ação, a cadeia causal e o
trace completo com a expectativa declarada antes da execução.

### Caso 1 — Estocada com vida baixa: conflito entre reação e cautela

**Entrada:** vida 20%, estocada, Mikiri desbloqueado, duas curas, sem abertura.
**Esperado:** `MIKIRI`; cadeia R2 → R5 → R12.
**Trace esperado:** R1, R2, R5, R10, R12.

R5 e R10 geram estratégias diferentes. R12 (30) vence R14 (10).
R1 e R10 aparecem no trace, mas não são antecedentes causais de Mikiri.
R9 não pode propor cura porque há ataque e não há abertura segura.

In [ ]:
caso1 = dict(cenario="C1 - Estocada com vida baixa", vida=20, ataque="estocada",
             mikiri=True, curas=2, abertura_segura=False)
resultado1 = executar_cenario(caso1)
# Esperado: reagir com Mikiri antes de escolher a cautela genérica.
assert resultado1["decisao"]["acao"] == "MIKIRI"
assert resultado1["decisao"]["cadeia"] == ("R2", "R5", "R12")
assert [e["regra"] for e in resultado1["trace"]] == ["R1", "R2", "R5", "R10", "R12"]
assert {c["regra"] for c in resultado1["conflitos"][0]["candidatas"]} == {"R12", "R14"}
print("TESTE 1: APROVADO")


CENÁRIO: C1 - Estocada com vida baixa
Entrada: {'cenario': 'C1 - Estocada com vida baixa', 'vida': 20, 'ataque': 'estocada', 'mikiri': True, 'curas': 2, 'abertura_segura': False}
[R1 | nível 1 | salience 100] A vida está no limiar de 30% ou abaixo dele. Conclusão: vida_baixa.
[R2 | nível 1 | salience 99] O movimento pertence ao grupo de ataques perigosos. Conclusão: ataque_perigoso.
[R5 | nível 2 | salience 90] É uma estocada e Mikiri está desbloqueado. Conclusão: MIKIRI.
[R10 | nível 2 | salience 85] A vida baixa também justifica uma alternativa de reposicionamento. Conclusão: BUSCAR_ABERTURA.
Candidatas à decisão:
  R12: MIKIRI (salience=30)
  R14: BUSCAR_ABERTURA (salience=10)
Selecionada pelo Experta: R12.
[R12 | nível 3 | salience 30] R12 seleciona a categoria reacao com salience 30. A presença da recomendação bloqueia outras decisões pelo NOT. Essa prioridade supera R14 (salience 10). Conclusão: MIKIRI.

AÇÃO: MIKIRI
Executar o Contra-ataque Mikiri no momento da estocada.
CADEIA

### Caso 2 — Vida baixa com cura segura: conflito entre cura e cautela

**Entrada:** vida 25%, nenhum ataque, duas curas e abertura segura.
**Esperado:** `CURAR`; cadeia R1 e R4 → R9 → R13.
**Trace esperado:** R1, R4, R9, R10, R13.

R9 combina dois fatos do primeiro nível. R13 (20) vence R14 (10).
`NOT(Recomendacao(...))` impede uma segunda decisão.

In [ ]:
caso2 = dict(cenario="C2 - Cura segura", vida=25, ataque="nenhum",
             mikiri=False, curas=2, abertura_segura=True)
resultado2 = executar_cenario(caso2)
# Esperado: usar a cura disponível, pois não há ataque e a abertura é segura.
assert resultado2["decisao"]["acao"] == "CURAR"
assert resultado2["decisao"]["cadeia"] == ("R1", "R4", "R9", "R13")
assert [e["regra"] for e in resultado2["trace"]] == ["R1", "R4", "R9", "R10", "R13"]
assert {c["regra"] for c in resultado2["conflitos"][0]["candidatas"]} == {"R13", "R14"}
print("TESTE 2: APROVADO")


CENÁRIO: C2 - Cura segura
Entrada: {'cenario': 'C2 - Cura segura', 'vida': 25, 'ataque': 'nenhum', 'mikiri': False, 'curas': 2, 'abertura_segura': True}
[R1 | nível 1 | salience 100] A vida está no limiar de 30% ou abaixo dele. Conclusão: vida_baixa.
[R4 | nível 1 | salience 97] Não há ataque em curso no cenário informado. Conclusão: sem_ataque.
[R9 | nível 2 | salience 86] Há vida baixa, carga de cura e abertura segura sem ataque. Conclusão: CURAR.
[R10 | nível 2 | salience 85] A vida baixa também justifica uma alternativa de reposicionamento. Conclusão: BUSCAR_ABERTURA.
Candidatas à decisão:
  R13: CURAR (salience=20)
  R14: BUSCAR_ABERTURA (salience=10)
Selecionada pelo Experta: R13.
[R13 | nível 3 | salience 20] R13 seleciona a categoria cura com salience 20. A presença da recomendação bloqueia outras decisões pelo NOT. Essa prioridade supera R14 (salience 10). Conclusão: CURAR.

AÇÃO: CURAR
Usar uma carga da Cabaça Curativa na abertura segura informada.
CADEIA CAUSAL: R1 + R4 → R

### Caso 3 — Vida no limite de 30% e nenhuma cura

**Entrada:** vida 30%, nenhum ataque, zero curas, sem abertura segura.
**Esperado:** `BUSCAR_ABERTURA`; cadeia R1 → R10 → R14.
**Trace esperado:** R1, R4, R10, R14.

O limite de 30% é inclusivo. R9 não dispara porque não há carga nem abertura;
R11 é bloqueada pela presença de vida baixa. A recomendação não promete
que reposicionar-se recupera vida ou cria cargas de cura.

In [ ]:
caso3 = dict(cenario="C3 - Vida baixa sem cura", vida=30, ataque="nenhum",
             mikiri=False, curas=0, abertura_segura=False)
resultado3 = executar_cenario(caso3)
# Esperado: buscar segurança, sem recomendar um recurso indisponível.
assert resultado3["decisao"]["acao"] == "BUSCAR_ABERTURA"
assert resultado3["decisao"]["cadeia"] == ("R1", "R10", "R14")
assert [e["regra"] for e in resultado3["trace"]] == ["R1", "R4", "R10", "R14"]
print("TESTE 3: APROVADO")


CENÁRIO: C3 - Vida baixa sem cura
Entrada: {'cenario': 'C3 - Vida baixa sem cura', 'vida': 30, 'ataque': 'nenhum', 'mikiri': False, 'curas': 0, 'abertura_segura': False}
[R1 | nível 1 | salience 100] A vida está no limiar de 30% ou abaixo dele. Conclusão: vida_baixa.
[R4 | nível 1 | salience 97] Não há ataque em curso no cenário informado. Conclusão: sem_ataque.
[R10 | nível 2 | salience 85] A vida baixa também justifica uma alternativa de reposicionamento. Conclusão: BUSCAR_ABERTURA.
Candidatas à decisão:
  R14: BUSCAR_ABERTURA (salience=10)
Selecionada pelo Experta: R14.
[R14 | nível 3 | salience 10] R14 seleciona a categoria cautela com salience 10. A presença da recomendação bloqueia outras decisões pelo NOT. Conclusão: BUSCAR_ABERTURA.

AÇÃO: BUSCAR_ABERTURA
Reposicionar-se para buscar segurança; reavaliar a cura depois.
CADEIA CAUSAL: R1 → R10 → R14
EXPLICAÇÃO: A ação BUSCAR_ABERTURA foi escolhida porque R1, R10, R14 dispararam. A vida está no limiar de 30% ou abaixo dele. A vid

## 9. Testes complementares e cobertura das regras

Os seis casos abaixo exercitam as respostas restantes, ambos os caminhos
de R7, o limite logo acima de 30% e a proibição de cura sem abertura.
As saídas resumidas mantêm a demonstração curta. O trace e a explicação
completos continuam disponíveis em `resultados_extras`.

In [ ]:
# (identificador, vida, ataque, mikiri, curas, abertura, ação esperada, cadeia esperada)
testes_extras = [
    ("C4 - Varredura", 80, "varredura", True, 2, False, "SALTAR", ("R2", "R6", "R12")),
    ("C5 - Agarre", 80, "agarre", True, 2, False, "AFASTAR", ("R2", "R7", "R12")),
    ("C6 - Sem Mikiri", 80, "estocada", False, 2, False, "AFASTAR", ("R2", "R7", "R12")),
    ("C7 - Ataque comum", 80, "normal", True, 2, False, "DEFLETIR", ("R3", "R8", "R12")),
    ("C8 - Acima de 30%", 30.1, "nenhum", False, 0, False, "OBSERVAR", ("R4", "R11", "R14")),
    ("C9 - Cura sem abertura", 20, "nenhum", True, 1, False, "BUSCAR_ABERTURA", ("R1", "R10", "R14")),
]
resultados_extras = []
for c, vida, ataque, mikiri, curas, abertura, esperado, cadeia in testes_extras:
    dados = dict(cenario=c, vida=vida, ataque=ataque, mikiri=mikiri,
                 curas=curas, abertura_segura=abertura)
    resultado = executar_cenario(dados, exibir=False)
    assert resultado["decisao"]["acao"] == esperado, c
    assert resultado["decisao"]["cadeia"] == cadeia, c
    resultados_extras.append(resultado)
    print(f"{c}: {esperado} | APROVADO")

resultados = [resultado1, resultado2, resultado3] + resultados_extras
regras_disparadas = {e["regra"] for r in resultados for e in r["trace"]}
regras_declaradas = {nome.upper() for nome, valor in vars(ConselheiroSekiro).items()
                    if isinstance(valor, Rule)}
assert regras_declaradas == {f"R{i}" for i in range(1, 15)}
assert regras_disparadas == regras_declaradas
for resultado in resultados:
    trace = resultado["trace"]
    cadeia = resultado["decisao"]["cadeia"]
    assert len([e for e in trace if e["nivel"] == 3]) == 1
    assert {e["nivel"] for e in trace if e["regra"] in cadeia} == {1, 2, 3}
    assert len(trace) == len({e["regra"] for e in trace})
    assert not resultado["motor"].agenda.activations
    for regra in cadeia:
        assert regra in resultado["decisao"]["explicacao"]

print(f"\n{len(resultados)} cenários aprovados; {len(regras_disparadas)}/14 regras exercitadas.")
print("Uma decisão por cenário, três níveis causais e nenhum disparo repetido.")

C4 - Varredura: SALTAR | APROVADO
C5 - Agarre: AFASTAR | APROVADO
C6 - Sem Mikiri: AFASTAR | APROVADO
C7 - Ataque comum: DEFLETIR | APROVADO
C8 - Acima de 30%: OBSERVAR | APROVADO
C9 - Cura sem abertura: BUSCAR_ABERTURA | APROVADO

9 cenários aprovados; 14/14 regras exercitadas.
Uma decisão por cenário, três níveis causais e nenhum disparo repetido.


### Validação de entradas fora do domínio

Os testes seguintes confirmam que morte, vida inválida, carga negativa e
abertura contraditória são rejeitadas antes da inferência.

In [ ]:
entradas_invalidas = [
    {**caso1, "vida": 0},
    {**caso1, "vida": float("nan")},
    {**caso1, "curas": -1},
    {**caso1, "abertura_segura": True},
]
for dados in entradas_invalidas:
    try:
        executar_cenario(dados, exibir=False)
    except ValueError as erro:
        print("REJEITADO corretamente:", erro)
    else:
        raise AssertionError("Uma entrada inválida foi aceita.")
print("4 verificações de entrada aprovadas.")

REJEITADO corretamente: vida deve ser um número finito maior que 0 e até 100.
REJEITADO corretamente: vida deve ser um número finito maior que 0 e até 100.
REJEITADO corretamente: curas deve ser um inteiro maior ou igual a zero.
REJEITADO corretamente: Abertura segura exige ausência de ataque em curso.
4 verificações de entrada aprovadas.


## 10. Experimentando outra situação

use
`resultado_personalizado["motor"].facts`; para consultar o histórico,
use `resultado_personalizado["trace"]`.

In [ ]:
meu_cenario = dict(
    cenario="Treino personalizado",
    vida=65,
    ataque="varredura",  # estocada, varredura, agarre, normal ou nenhum
    mikiri=True,
    curas=1,
    abertura_segura=False,
)
resultado_personalizado = executar_cenario(meu_cenario)


CENÁRIO: Treino personalizado
Entrada: {'cenario': 'Treino personalizado', 'vida': 65, 'ataque': 'varredura', 'mikiri': True, 'curas': 1, 'abertura_segura': False}
[R2 | nível 1 | salience 99] O movimento pertence ao grupo de ataques perigosos. Conclusão: ataque_perigoso.
[R6 | nível 2 | salience 89] A varredura requer uma resposta por salto neste modelo. Conclusão: SALTAR.
Candidatas à decisão:
  R12: SALTAR (salience=30)
Selecionada pelo Experta: R12.
[R12 | nível 3 | salience 30] R12 seleciona a categoria reacao com salience 30. A presença da recomendação bloqueia outras decisões pelo NOT. Conclusão: SALTAR.

AÇÃO: SALTAR
Saltar sobre a varredura e, se possível, saltar sobre o inimigo.
CADEIA CAUSAL: R2 → R6 → R12
EXPLICAÇÃO: A ação SALTAR foi escolhida porque R2, R6, R12 dispararam. O movimento pertence ao grupo de ataques perigosos. A varredura requer uma resposta por salto neste modelo. R12 seleciona a categoria reacao com salience 30. A presença da recomendação bloqueia outras 

## 11. Discussão dos resultados e limitações

Os testes mostram que a mesma base combina interpretação, estratégia e
decisão. No primeiro caso, reagir à estocada tem prioridade sobre buscar
abertura; no segundo, a cura segura vence a cautela; no terceiro, a ausência
de recursos impede uma recomendação de cura. As explicações são obtidas
das regras e dos fatos realmente utilizados.

Os nove cenários exercitam todas as 14 regras, incluindo os dois ramos de
R7. Isso verifica a implementação desses casos; não mede eficácia em
partidas nem demonstra que a política é ótima para todo inimigo.

O sistema não reconhece imagens, não aprende com partidas e não controla
o jogo. Não modela postura, distância numérica, timing, ferramentas
prostéticas, múltiplos inimigos, ressurreição ou exceções de chefes.
`abertura_segura` depende da avaliação do usuário. O limiar e a ordem das
ações são heurísticas didáticas, justificadas pela necessidade de produzir
uma recomendação imediata e explicável.

Uma extensão possível seria acrescentar postura e distância aos fatos,
com regras específicas e novos testes. O conhecimento continuaria separado
do motor de inferência, facilitando essa evolução.